# NumPy 学习笔记（入门速通）

> 目标：掌握 `ndarray` 的创建、索引、形状变换、广播、向量化运算与常用统计 / 线性代数操作。

NumPy（Numerical Python）是科学计算生态的地基：`pandas`、`scipy`、`matplotlib`、`scikit-learn` 的底层数据结构都建立在 `ndarray` 之上。它的最大优势是**向量化运算**——同样的计算用 NumPy 写成数组表达式，比 Python 原生 `for` 循环快 1~2 个数量级。

**环境准备**
```bash
pip install numpy
```

In [10]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## 1. 数组创建

NumPy 的核心对象是 `ndarray`（N 维数组）。创建方式大致分三类：**从 Python 序列转换**、**按规则生成**、**特殊结构 / 随机生成**。

| 函数 | 作用 | 关键参数与说明 |
|---|---|---|
| `np.array(obj, dtype=None)` | 从 list / tuple / 嵌套序列构造数组 | `dtype` 指定元素类型，省略时自动推断 |
| `np.arange(start, stop, step, dtype=None)` | 类似内置 `range`，生成等差序列 | **不包含** `stop`，支持浮点步长 |
| `np.linspace(start, stop, num=50, endpoint=True)` | 在区间内生成 `num` 个等距点 | 默认**包含** `stop`；`endpoint=False` 可排除 |
| `np.logspace(start, stop, num)` | 生成以 10 为底的对数刻度序列 | 常用于画频谱、学习率网格 |
| `np.zeros(shape)` / `np.ones(shape)` | 全 0 / 全 1 数组 | `shape` 传元组；`dtype` 默认 `float64` |
| `np.full(shape, fill_value)` | 用指定常量填充 | 适合初始化权重、掩码 |
| `np.eye(N, M=None, k=0)` | 单位矩阵 / 对角矩阵 | `k` 控制对角线偏移：`k>0` 上移，`k<0` 下移 |
| `np.empty(shape)` | 只分配内存、不初始化 | 速度最快，但内容是**垃圾值**，必须自己覆盖 |

**易错点**
- `arange` 不含终点、`linspace` 默认含终点，两者语义相反，初学最容易混。
- `np.zeros` / `np.ones` 默认是浮点型，需要整数时显式写 `dtype=int`。
- `np.empty` 不是「全 0 数组」，它的值不确定，不能直接当初始值使用。

In [11]:
a = np.array([1, 2, 3])
b = np.arange(0, 10, 2)          # [0 2 4 6 8]
c = np.linspace(0, 1, 5)         # [0. 0.25 0.5 0.75 1.]
d = np.zeros((2, 3))
e = np.ones((2, 2), dtype=np.int32)
f = np.full((2, 2), 7)
g = np.eye(3, k=1)               # 对角线向上偏移 1
h = np.empty((2, 2))             # 未初始化，值不确定
print(a, b, c, d, e, f, g, h, sep="\n---\n")

[1 2 3]
---
[0 2 4 6 8]
---
[0.   0.25 0.5  0.75 1.  ]
---
[[0. 0. 0.]
 [0. 0. 0.]]
---
[[1 1]
 [1 1]]
---
[[7 7]
 [7 7]]
---
[[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
---
[[ 1.  4.]
 [ 9. 16.]]


## 2. 数组属性

属性用来「看」数组的元信息，调试和形状对齐时最常用。

| 属性 | 含义 |
|---|---|
| `shape` | 各维度大小组成的元组，如 `(3, 4)` |
| `ndim` | 维度数（轴的数量），等于 `len(shape)` |
| `dtype` | 元素数据类型，如 `int64`、`float32` |
| `size` | 元素总个数，等于 `shape` 各维相乘 |
| `itemsize` | 单个元素占用的字节数 |
| `nbytes` | 整个数组占用的字节数，等于 `size * itemsize` |
| `T` | 转置视图（维度反转） |

**类型转换**用 `astype(dtype)`：它会返回**新数组**（默认复制），并做类型转换；`dtype` 可用字符串 `'float32'` 或 `np.float32`。

**易错点**：`shape`、`dtype` 是属性不是方法，写 `x.shape` 而不是 `x.shape()`。

In [12]:
x = np.arange(12).reshape(3, 4)
print("shape   :", x.shape)     # (3, 4)
print("ndim    :", x.ndim)      # 2
print("dtype   :", x.dtype)     # int64
print("size    :", x.size)      # 12
print("itemsize:", x.itemsize)  # 8
print("nbytes  :", x.nbytes)    # 96

y = x.astype(np.float32)
print("astype  :", y.dtype)     # float32

shape   : (3, 4)
ndim    : 2
dtype   : int32
size    : 12
itemsize: 4
nbytes  : 48
astype  : float32


## 3. 索引与切片

NumPy 提供三种主要索引方式，理解它们的「视图 / 副本」差异非常关键。

**① 基础索引 / 切片**：`x[i, j]`、`x[start:stop:step]`、`x[..., 0]`。多维用逗号分隔各轴，`:` 表示整轴，`...` 表示省略的中间轴。

**② 布尔索引**：`x[x > 5]`。先由比较运算得到同形状的布尔数组（掩码），再用它筛选元素，返回**一维**结果。
- 多个条件组合要用 `&`、`|`、`~`，且每个条件必须**加括号**：`(x > 3) & (x < 9)`。

**③ 花式索引（fancy indexing）**：用整数数组当索引，如 `x[[0, 2], [1, 3]]`，会按位置一一配对取值，返回**副本**。
- 想对多维同时取行列，可借助 `np.ix_` 构造外积式索引。

**视图 vs 副本**
- 基础切片返回**视图**：修改它会影响原数组，不额外占内存。
- 布尔索引、花式索引返回**副本**：修改不影响原数组。
- 想要独立副本时用 `x[...].copy()`。

In [13]:
x = np.arange(12).reshape(3, 4)
print(x[0, 1])            # 标量 1
print(x[:, 1])            # 第 1 列
print(x[1:3, ::2])        # 行 1~2，列步长 2
print(x[x > 5])           # 布尔索引
print(x[[0, 2], [1, 3]])  # 花式索引 -> [1 11]

m = (x > 3) & (x < 9)     # 多条件要加括号
print("mask 筛选:", x[m])

v = x[:, 0]
v[0] = 999                # 切片是视图，会修改原数组
print("视图修改后 x[0,0] =", x[0, 0])
print("花式索引是副本:", x[[0, 2], [1, 3]].base is None)

1
[1 5 9]
[[ 4  6]
 [ 8 10]]
[ 6  7  8  9 10 11]
[ 1 11]
mask 筛选: [4 5 6 7 8]
视图修改后 x[0,0] = 999
花式索引是副本: True


## 4. 形状变换

| 函数 | 作用 | 说明 |
|---|---|---|
| `reshape(shape)` | 改变形状 | 支持 `-1` 自动推断某一维；元素总数必须一致 |
| `ravel()` | 展平成一维 | **尽量返回视图**，不复制 |
| `flatten()` | 展平成一维 | **总是返回副本** |
| `transpose()` / `.T` | 转置 / 维度反转 | 高维可用 `swapaxes(a, axis1, axis2)` |
| `expand_dims(a, axis)` | 在指定位置插入长度 1 的维度 | 广播前常用 |
| `squeeze(a, axis=None)` | 删除长度为 1 的维度 | 不指定 `axis` 时删除所有长度为 1 的轴 |
| `concatenate([a, b], axis)` | 沿已有轴拼接 | 其它维度必须一致 |
| `stack([a, b], axis)` | 沿**新轴**堆叠 | 会多出一个维度 |

**易错点**
- `reshape` 不会改原数组，而是返回新视图；想原地改形状可给 `shape` 赋值（要求内存连续）。
- `concatenate` 与 `stack` 的区别：前者拼在现有轴上，后者新增一个轴。
- `ravel()` 返回视图还是副本取决于内存是否连续，不确定时用 `flatten()`。

In [14]:
a = np.arange(6)
print(a.reshape(2, 3))
print(a.reshape(2, -1))                  # -1 自动推断为 3
print(a.reshape(2, 3).T)                 # 转置
print(np.expand_dims(a, axis=0).shape)   # (1, 6)
print(np.squeeze(np.zeros((1, 3, 1))).shape)  # (3,)
print(np.concatenate([a, a]))            # 沿轴 0 拼接
print(np.stack([a, a], axis=1).shape)    # 新轴堆叠 -> (6, 2)

r = a.reshape(2, 3)
print("ravel 是否视图:", r.ravel().base is not None)
print("flatten 是否副本:", r.flatten().base is None)

[[0 1 2]
 [3 4 5]]
[[0 1 2]
 [3 4 5]]
[[0 3]
 [1 4]
 [2 5]]
(1, 6)
(3,)
[0 1 2 3 4 5 0 1 2 3 4 5]
(6, 2)
ravel 是否视图: True
flatten 是否副本: True


## 5. 广播机制（Broadcasting）

广播让**形状不同**的数组也能做逐元素运算，是 NumPy 最核心的便利之一，也是很多报错的来源。

**规则（从尾部维度开始逐位比较）**
1. 维度数不同时，在**较小数组形状的前面补 1**（左补）。
2. 某一维上两者大小相等，或其中一个为 1 → 兼容，大小为 1 的那一维会被「拉伸」复制。
3. 以上都不满足 → 抛出 `ValueError`。

**典型例子**
- `(2, 3) + (3,)`：右侧补成 `(1, 3)`，再拉伸到 `(2, 3)` ✅
- `(2, 3) + (2, 1)`：第二维 1 → 3，得到 `(2, 3)` ✅
- `(2, 3) + (2,)`：尾部 3 vs 2 不兼容 ❌

**相关工具**
- `np.broadcast_to(a, shape)`：把数组广播成目标形状（返回只读视图）。
- `np.broadcast_arrays(a, b)`：返回广播后的多个数组。

**易错点**：广播是「逻辑上的复制」，不会真的占用对应内存；但若对广播结果写入会报只读错误，需要 `.copy()`。

In [15]:
a = np.arange(6).reshape(2, 3)
b = np.array([10, 20, 30])
print(a + b)                     # (2,3) + (3,)   -> (2,3)

col = np.array([[100], [200]])
print(a + col)                   # (2,3) + (2,1)  -> (2,3)

print(a + 5)                     # 标量广播
print(np.broadcast_to(b, (2, 3)))

try:
    a + np.array([1, 2])
except ValueError as e:
    print("广播失败:", e)

[[10 21 32]
 [13 24 35]]
[[100 101 102]
 [203 204 205]]
[[ 5  6  7]
 [ 8  9 10]]
[[10 20 30]
 [10 20 30]]
广播失败: operands could not be broadcast together with shapes (2,3) (2,) 


## 6. 通用函数（ufunc）

ufunc（universal function）是**对数组逐元素运算、底层用 C 实现**的函数，是向量化的主力。

| 类别 | 常用函数 |
|---|---|
| 算术 | `add`、`subtract`、`multiply`、`divide`、`floor_divide`、`mod`、`power` |
| 数学 | `sqrt`、`exp`、`log`、`log10`、`sin`、`cos`、`abs`、`sign`、`ceil`、`floor` |
| 比较 | `greater`、`less`、`equal`、`logical_and`、`logical_or` |
| 聚合方法 | `ufunc.reduce`（归约）、`accumulate`（累加）、`outer`（外积） |

**重要参数**
- `out=`：把结果写进已有数组，避免反复分配内存。
- `where=`：条件掩码，只对满足条件的位置计算。

**为什么快**：NumPy 把循环下沉到 C 层并利用 SIMD，同时避开 Python 对象开销。原则是「**能用数组表达式，就别写 for 循环**」。

In [16]:
a = np.array([1, 4, 9, 16], dtype=float)
print(np.sqrt(a))
print(np.exp([0, 1]))
print(np.sin(np.pi / 2))
print(np.abs(np.array([-1, 2, -3])))
print(np.add(a, 1))
print(np.power(a, 2))
print(np.greater(a, 5))          # 返回布尔数组

out = np.empty_like(a)
np.add(a, 1, out=out)            # 结果写入已有数组
print("out:", out)

[1. 2. 3. 4.]
[1.         2.71828183]
1.0
[1 2 3]
[ 2.  5. 10. 17.]
[  1.  16.  81. 256.]
[False False  True  True]
out: [ 2.  5. 10. 17.]


## 7. 聚合与统计

聚合函数把数组「压缩」成更少的元素，`axis` 参数决定沿哪一维压缩。

| 函数 | 作用 |
|---|---|
| `sum` / `prod` | 求和 / 求积 |
| `mean` / `median` / `std` / `var` | 均值 / 中位数 / 标准差 / 方差 |
| `min` / `max` | 最小 / 最大值 |
| `argmin` / `argmax` / `argsort` | 最值下标 / 排序后的下标 |
| `percentile(a, q)` | 分位数 |
| `np.where(cond, x, y)` | 条件选择（向量化 if-else） |
| `any` / `all` / `count_nonzero` | 是否存在真值 / 是否全真 / 非零计数 |

**理解 `axis`**：`axis=0` 表示「沿第 0 维（行方向）压缩」，结果形状去掉第 0 维，也就是**每列**的统计量；`axis=1` 得到**每行**的统计量。

**`keepdims=True`** 保留被压缩的维度（长度变为 1），便于后续广播，例如 `x / x.sum(axis=1, keepdims=True)` 做行归一化。

**易错点**：`x.sum()` 是方法，`np.sum(x)` 是函数，两者等价；空数组求 `mean` 会返回 `nan` 并给出警告。

In [ ]:
x = np.arange(12).reshape(3, 4)
print("sum all   :", x.sum())
print("sum axis=0:", x.sum(axis=0))     # 每列
print("mean axis=1:", x.mean(axis=1))   # 每行
print("std       :", x.std())
print("median    :", np.median(x))
print("argmax    :", x.argmax())
print("where     :", np.where(x > 6, 1, 0))
print("any >10   :", (x > 10).any())
print("all >=0   :", (x >= 0).all())

row_norm = x / x.sum(axis=1, keepdims=True)
print("行归一化:\n", row_norm)

## 8. 线性代数

线性代数函数集中在 `np.linalg` 子模块，矩阵乘法也可直接用 `@` 运算符。

| 函数 | 作用 |
|---|---|
| `np.dot(a, b)` / `a @ b` | 点积 / 矩阵乘法 |
| `np.matmul(a, b)` | 批量矩阵乘法（高维按最后两维成批运算） |
| `np.linalg.inv(A)` | 求逆矩阵 |
| `np.linalg.det(A)` | 行列式 |
| `np.linalg.eig(A)` / `eigvals(A)` | 特征值 + 特征向量 / 仅特征值 |
| `np.linalg.solve(A, b)` | 解线性方程组 `Ax = b`（比求逆再乘更快更稳） |
| `np.linalg.lstsq(A, b)` | 最小二乘解（超定方程） |
| `np.linalg.svd(A)` | 奇异值分解 |
| `np.linalg.norm(x)` | 向量 / 矩阵范数 |

**`dot` vs `matmul`**：二维时两者结果相同；高维时 `dot` 按「a 的最后一维 × b 的倒数第二维」求和，而 `matmul` 把前面的维度当批次、只对最后两维做矩阵乘，更符合深度学习的直觉。

**易错点**：`*` 是**逐元素乘**，矩阵乘法必须用 `@` 或 `dot`；数值求解优先用 `solve` 而不是 `inv(A) @ b`。

In [ ]:
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])
print("A @ B:\n", A @ B)
print("A * B (逐元素):\n", A * B)
print("inv(A):\n", np.linalg.inv(A))
print("det(A):", np.linalg.det(A))
print("eigvals:", np.linalg.eigvals(A))

b = np.array([1., 2.])
xs = np.linalg.solve(A, b)
print("solve Ax=b:", xs)
print("norm b:", np.linalg.norm(b))

## 9. 随机数

新式 API 通过 `np.random.default_rng(seed)` 创建 **Generator** 对象，替代旧的全局 `np.random.*` 接口（旧接口仍可用但不推荐，且不利于并行与复现）。

| 方法 | 作用 |
|---|---|
| `random(size)` | `[0, 1)` 均匀分布 |
| `uniform(low, high, size)` | 指定区间均匀分布 |
| `normal(loc, scale, size)` | 正态分布（均值、标准差） |
| `integers(low, high, size)` | 随机整数，**不含** `high` |
| `choice(a, size, replace=True)` | 从 `a` 中抽样，`replace=False` 为不放回 |
| `permutation(n)` | 返回打乱后的序列（不改原数组） |
| `shuffle(a)` | **原地**打乱数组 |

**可复现性**：固定 `seed` 后，同一 Generator 上连续调用会得到确定性序列，方便调试和实验对比。

**易错点**：`choice` 默认有放回，想不重复抽样必须写 `replace=False`，且 `size` 不能超过总体大小。

In [ ]:
rng = np.random.default_rng(seed=42)
print(rng.random((2, 3)))              # [0,1) 均匀
print(rng.normal(0, 1, size=5))        # 标准正态
print(rng.integers(0, 10, size=5))     # 随机整数
print(rng.uniform(0, 1, size=3))       # 均匀分布

a = np.arange(5)
rng.shuffle(a)                         # 原地打乱
print("shuffled:", a)
print("choice:", rng.choice(a, size=3, replace=False))
print("permutation:", rng.permutation(5))

## 10. 综合练习

1. 生成 `100x3` 的标准正态矩阵，计算**每列**的均值与标准差。
2. 给定矩阵 `X`，把大于均值的元素置为 1、其余为 0（要求向量化实现，不使用循环）。
3. 用 `np.linalg.solve` 求解一个 3x3 线性方程组，并用 `np.allclose` 验证 `A @ x ≈ b`。

**参考答案**

In [17]:
rng = np.random.default_rng(0)
M = rng.normal(size=(100, 3))
print("mean:", M.mean(axis=0))
print("std :", M.std(axis=0))

X = np.arange(9).reshape(3, 3)
Y = np.where(X > X.mean(), 1, 0)
print(Y)

A = np.array([[3., 1., 0.], [1., 2., 1.], [0., 1., 3.]])
b = np.array([9., 8., 9.])
xs = np.linalg.solve(A, b)
print("x =", xs, " 验证:", np.allclose(A @ xs, b))

mean: [-0.10241404  0.00340769 -0.00844442]
std : [1.08768754 1.00332574 0.95468345]
[[0 0 0]
 [0 0 1]
 [1 1 1]]
x = [2.5 1.5 2.5]  验证: True


## 常见坑点总结

- **切片是视图**，修改会影响原数组；需要独立数据用 `.copy()`。布尔索引和花式索引返回副本。
- **`arange` 不含终点，`linspace` 含终点**，写循环时尤其注意边界。
- **整数除法会截断**：`np.array([1, 2]) / 2` 结果还是浮点没问题，但 `//` 是整除；需要小数时先 `astype(float)`。
- **`axis` 含义**：`axis=0` 沿行方向压缩（得到每列统计），`axis=1` 得到每行统计。
- **广播失败**多半是尾部维度不对齐，用 `reshape` / `expand_dims` 修正后再运算。
- **矩阵乘法用 `@`**，`*` 是逐元素乘，两者语义完全不同。
- **`np.empty` 不是零数组**，内容不确定，必须先赋值再用。
- **性能**：优先写向量化表达式；确实需要循环时考虑 `np.vectorize`（仅语法糖，不快）或 `numba`。